# 과제 3. 데이터 시각화

이번 과제에서는 두 개의 데이터셋을 사용하여 데이터의 특징을 시각화하였다.

첫 번째 데이터셋은 스마트폰 사용 및 중독 분석 데이터셋이다. 이 데이터는 7,500명의 사용자에 대해 스마트폰 사용 시간, SNS 사용 시간, 게임 시간, 수면 시간, 알림 수, 앱 실행 횟수, 스트레스 수준, 중독 여부 등을 포함한다. 이 데이터는 사용자의 스마트폰 사용 습관과 중독 지표 사이의 관계를 확인하는 데 적합하다.

두 번째 데이터셋은 중고차 가격 데이터셋이다. 이 데이터는 8,128대의 차량에 대해 차량명, 제조연도, 주행거리, 연료 종류, 판매자 유형, 변속기, 소유자 유형, 연비, 엔진, 좌석 수, 판매가격 등을 포함한다. 중고차 가격은 여러 조건에 따라 달라지므로, Bokeh를 이용하여 사용자가 직접 조건을 바꾸며 확인할 수 있는 동적 시각화를 구성하였다.

## 0. 라이브러리 및 데이터 불러오기

데이터 파일은 노트북과 같은 폴더의 `data` 폴더 안에 넣었다. 그래프에서 한글이 깨지지 않도록 기본 폰트를 설정하였다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

DATA_DIR = Path('data')

smartphone_path = DATA_DIR / 'Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv'
car_path = DATA_DIR / 'Car_Price.csv'

smart_df = pd.read_csv(smartphone_path)
car_df = pd.read_csv(car_path)

print('스마트폰 데이터 크기:', smart_df.shape)
print('중고차 데이터 크기:', car_df.shape)

display(smart_df.head())
display(car_df.head())

In [ ]:
print('스마트폰 데이터 컬럼')
print(smart_df.columns.tolist())

print('중고차 데이터 컬럼')
print(car_df.columns.tolist())

## 1. 스마트폰 사용 및 중독 분석 데이터 시각화 설계

### 전체 시각화 목적

스마트폰 데이터 시각화의 목적은 사용자의 스마트폰 사용 습관이 중독 여부, 수면 시간, 스트레스 수준, 학업 및 업무 영향과 어떤 관계가 있는지 확인하는 것이다. 단순히 사용 시간이 긴 사람을 찾는 것이 아니라, 사용 시간과 생활 패턴 사이의 관계를 그래프로 확인하는 데 초점을 두었다.

### 시각화 그래프 설계

스마트폰 데이터는 사용자별 행으로 구성되어 있으므로, 먼저 주요 변수의 분포를 확인하고 이후 변수 간 관계를 비교하는 방식으로 설계하였다. 사용한 그래프는 히스토그램, 막대그래프, 박스플롯, 산점도, 회귀선 그래프, 히트맵이다.

### 시각화를 통해 얻을 정보

이 시각화를 통해 스마트폰 사용 시간이 많은 사용자의 특징, 중독 여부에 따른 사용 패턴 차이, 수면 시간과 사용 시간의 관계, 스트레스 수준이나 학업/업무 영향과 중독률의 관계를 확인할 수 있다.

In [ ]:
# 분석에 필요한 파생 컬럼 생성
smart = smart_df.copy()

smart['addiction_level_filled'] = smart['addiction_level'].fillna('None')
smart['addicted_text'] = smart['addicted_label'].map({0: 'Not addicted', 1: 'Addicted'})

smart['age_group'] = pd.cut(
    smart['age'],
    bins=[0, 19, 29, 39, 49, 59, 100],
    labels=['10대 이하', '20대', '30대', '40대', '50대', '60대 이상']
)

level_order = ['None', 'Mild', 'Moderate', 'Severe']
stress_order = ['Low', 'Medium', 'High']
impact_order = ['No', 'Yes']

smart.head()

### 그래프 1. 일일 스마트폰 사용 시간 분포

이 그래프의 목적은 전체 사용자의 하루 스마트폰 사용 시간이 어느 구간에 많이 분포하는지 확인하는 것이다. X축은 하루 스마트폰 사용 시간, Y축은 사용자 수를 의미한다. 평균선을 함께 표시하여 전체 사용자의 일반적인 사용 수준도 확인할 수 있도록 하였다.

이 그래프를 통해 대부분의 사용자가 어느 정도의 시간을 스마트폰에 사용하는지, 그리고 사용 시간이 매우 긴 사용자가 존재하는지 확인할 수 있다.

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(smart['daily_screen_time_hours'], bins=30, alpha=0.8, edgecolor='black')
mean_value = smart['daily_screen_time_hours'].mean()
plt.axvline(mean_value, color='red', linestyle='--', label=f'평균: {mean_value:.2f}시간')

plt.title('일일 스마트폰 사용 시간 분포')
plt.xlabel('일일 스마트폰 사용 시간(시간)')
plt.ylabel('사용자 수')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 그래프 2. 스마트폰 중독 수준별 사용자 수

이 그래프의 목적은 스마트폰 중독 수준이 데이터 안에서 어떻게 나누어져 있는지 확인하는 것이다. X축은 중독 수준, Y축은 해당 수준에 속하는 사용자 수를 의미한다. `None`은 중독 수준이 따로 기록되지 않은 사용자를 의미한다.

중독 수준별 사용자 수를 보면 이 데이터에서 가벼운 중독, 중간 수준 중독, 심한 중독 사용자가 어느 정도 비율로 존재하는지 파악할 수 있다.

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=smart, x='addiction_level_filled', order=level_order, color='C0')

plt.title('스마트폰 중독 수준별 사용자 수')
plt.xlabel('스마트폰 중독 수준')
plt.ylabel('사용자 수')

for container in ax.containers:
    ax.bar_label(container)

plt.grid(True, axis='y', alpha=0.3)
plt.show()

### 그래프 3. 중독 수준에 따른 일일 스마트폰 사용 시간 비교

이 그래프의 목적은 중독 수준이 높아질수록 실제 스마트폰 사용 시간이 길어지는지 비교하는 것이다. X축은 중독 수준, Y축은 일일 스마트폰 사용 시간이다. 박스플롯은 중앙값, 사분위 범위, 이상치를 함께 보여주기 때문에 집단 간 차이를 비교하기 좋다.

중독 수준이 높을수록 박스가 위쪽에 위치한다면, 스마트폰 중독 수준과 사용 시간이 관련이 있다는 의미로 해석할 수 있다.

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=smart, x='addiction_level_filled', y='daily_screen_time_hours', order=level_order, color='C0')

plt.title('중독 수준에 따른 일일 스마트폰 사용 시간 비교')
plt.xlabel('스마트폰 중독 수준')
plt.ylabel('일일 스마트폰 사용 시간(시간)')
plt.grid(True, axis='y', alpha=0.3)
plt.show()

### 그래프 4. 스마트폰 사용 시간과 수면 시간의 관계

이 그래프의 목적은 스마트폰을 오래 사용할수록 수면 시간이 줄어드는 경향이 있는지 확인하는 것이다. X축은 일일 스마트폰 사용 시간, Y축은 수면 시간이며, 회귀선을 함께 표시하였다.

회귀선이 아래로 기울어진다면 스마트폰 사용 시간이 길수록 수면 시간이 감소하는 경향이 있다고 볼 수 있다. 점들이 넓게 퍼져 있다면 개인별 차이도 크다는 의미이다.

In [ ]:
plt.figure(figsize=(9, 6))
sns.regplot(
    data=smart,
    x='daily_screen_time_hours',
    y='sleep_hours',
    scatter_kws={'alpha': 0.35},
    line_kws={'color': 'red'}
)

plt.title('스마트폰 사용 시간과 수면 시간의 관계')
plt.xlabel('일일 스마트폰 사용 시간(시간)')
plt.ylabel('수면 시간(시간)')
plt.grid(True, alpha=0.3)
plt.show()

### 그래프 5. 스트레스 수준별 스마트폰 중독 비율

이 그래프의 목적은 스트레스 수준에 따라 스마트폰 중독 비율이 달라지는지 확인하는 것이다. X축은 스트레스 수준, Y축은 중독 사용자 비율이다. 여기서 중독 비율은 `addicted_label`의 평균으로 계산하였다.

스트레스 수준이 높을수록 중독 비율도 높게 나타난다면, 스트레스와 스마트폰 의존 사이에 관련성이 있을 가능성을 생각해 볼 수 있다.

In [ ]:
stress_addiction = (
    smart.groupby('stress_level')['addicted_label']
    .mean()
    .reindex(stress_order)
    .reset_index()
)
stress_addiction['addiction_rate_percent'] = stress_addiction['addicted_label'] * 100

plt.figure(figsize=(8, 5))
ax = sns.barplot(data=stress_addiction, x='stress_level', y='addiction_rate_percent', order=stress_order, color='C0')

plt.title('스트레스 수준별 스마트폰 중독 비율')
plt.xlabel('스트레스 수준')
plt.ylabel('중독 비율(%)')
plt.ylim(0, 100)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

### 그래프 6. 성별에 따른 일일 스마트폰 사용 시간 비교

이 그래프의 목적은 성별에 따라 스마트폰 사용 시간의 차이가 있는지 확인하는 것이다. X축은 성별, Y축은 일일 스마트폰 사용 시간이다.

박스플롯을 사용하면 성별별 중앙값과 데이터의 퍼짐 정도를 함께 볼 수 있다. 특정 성별의 박스가 더 높게 나타나면 해당 집단의 스마트폰 사용 시간이 상대적으로 긴 편이라고 해석할 수 있다.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=smart, x='gender', y='daily_screen_time_hours', color='C0')

plt.title('성별에 따른 일일 스마트폰 사용 시간 비교')
plt.xlabel('성별')
plt.ylabel('일일 스마트폰 사용 시간(시간)')
plt.grid(True, axis='y', alpha=0.3)
plt.show()

### 그래프 7. 연령대별 스마트폰 중독 비율

이 그래프의 목적은 연령대에 따라 스마트폰 중독 비율이 어떻게 달라지는지 확인하는 것이다. X축은 연령대, Y축은 중독 비율이다.

연령대별 중독 비율을 비교하면 특정 나이대에서 스마트폰 의존 경향이 더 강하게 나타나는지 확인할 수 있다.

In [ ]:
age_addiction = (
    smart.groupby('age_group', observed=False)['addicted_label']
    .mean()
    .reset_index()
)
age_addiction['addiction_rate_percent'] = age_addiction['addicted_label'] * 100

plt.figure(figsize=(9, 5))
ax = sns.barplot(data=age_addiction, x='age_group', y='addiction_rate_percent', color='C0')

plt.title('연령대별 스마트폰 중독 비율')
plt.xlabel('연령대')
plt.ylabel('중독 비율(%)')
plt.ylim(0, 100)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

### 그래프 8. 주요 수치형 변수 간 상관관계

이 그래프의 목적은 스마트폰 사용 관련 수치형 변수들이 서로 어떤 관계를 가지는지 한 번에 확인하는 것이다. 히트맵의 값은 상관계수이며, 1에 가까울수록 양의 관계가 강하고 -1에 가까울수록 음의 관계가 강하다는 뜻이다.

이 그래프를 통해 스마트폰 사용 시간, SNS 사용 시간, 게임 시간, 수면 시간, 알림 수, 앱 실행 횟수 등이 중독 여부와 어떤 방향의 관계를 가지는지 확인할 수 있다.

In [ ]:
numeric_cols = [
    'age',
    'daily_screen_time_hours',
    'social_media_hours',
    'gaming_hours',
    'work_study_hours',
    'sleep_hours',
    'notifications_per_day',
    'app_opens_per_day',
    'weekend_screen_time',
    'addicted_label'
]

corr = smart[numeric_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)

plt.title('스마트폰 데이터 주요 수치형 변수 간 상관관계')
plt.xlabel('변수')
plt.ylabel('변수')
plt.show()

### 그래프 9. 학업/업무 영향 여부에 따른 평균 스마트폰 사용 시간

이 그래프의 목적은 스마트폰 사용이 학업이나 업무에 영향을 준다고 응답한 집단과 그렇지 않은 집단의 평균 사용 시간을 비교하는 것이다. X축은 학업/업무 영향 여부, Y축은 평균 스마트폰 사용 시간이다.

영향이 있다고 응답한 집단의 평균 사용 시간이 더 높게 나타난다면, 스마트폰 사용 시간이 실제 생활의 집중도나 생산성에 영향을 줄 가능성이 있다고 해석할 수 있다.

In [ ]:
impact_screen = (
    smart.groupby('academic_work_impact')['daily_screen_time_hours']
    .mean()
    .reindex(impact_order)
    .reset_index()
)

plt.figure(figsize=(7, 5))
ax = sns.barplot(data=impact_screen, x='academic_work_impact', y='daily_screen_time_hours', order=impact_order, color='C0')

plt.title('학업/업무 영향 여부에 따른 평균 스마트폰 사용 시간')
plt.xlabel('학업/업무 영향 여부')
plt.ylabel('평균 일일 스마트폰 사용 시간(시간)')

for container in ax.containers:
    ax.bar_label(container, fmt='%.2f')

plt.grid(True, axis='y', alpha=0.3)
plt.show()

## 2. 중고차 가격 데이터 시각화 설계

### 전체 시각화 목적

중고차 가격 데이터의 시각화 목적은 차량 가격이 제조연도, 주행거리, 연료 종류, 변속기, 소유자 유형 등에 따라 어떻게 달라지는지 확인하는 것이다. 중고차 가격은 여러 조건이 동시에 영향을 주기 때문에 정적인 그래프만으로는 모든 경우를 보기 어렵다.

따라서 Bokeh를 이용하여 사용자가 드롭다운과 슬라이더를 직접 조작하면서 원하는 조건의 차량만 확인할 수 있도록 동적 시각화를 설계하였다.

### 시각화 그래프 설계

중고차 데이터는 차량별 가격과 차량 속성이 함께 존재하므로, 조건별 필터링이 가능한 그래프를 중심으로 구성하였다. 제조사, 연료 종류, 제조연도, 변속기, 소유자 유형, 가격 범위를 기준으로 동적 그래프를 만들었다.

### 시각화를 통해 얻을 정보

이 시각화를 통해 연식이 최신일수록 가격이 높은지, 주행거리가 길수록 가격이 낮아지는지, 연료 종류나 변속기에 따라 가격 차이가 존재하는지 확인할 수 있다. 또한 특정 제조사나 가격대의 차량만 선택하여 더 자세히 살펴볼 수 있다.

## 2-1. Bokeh 사용을 위한 중고차 데이터 전처리

중고차 가격은 원 단위로 되어 있어 그래프에서 보기 쉽도록 `Lakh` 단위로 변환하였다. 또한 차량명에서 첫 번째 단어를 추출하여 제조사 컬럼을 만들었다.

In [ ]:
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import column
from bokeh.models import ColumnDataSource, Select, RangeSlider, CustomJS, HoverTool, Div

output_notebook()

car = car_df.copy()

car['brand'] = car['name'].astype(str).str.split().str[0]
car['price_lakh'] = car['selling_price'] / 100000
car['car_age'] = 2026 - car['year']

# Bokeh에서 안전하게 사용하기 위해 문자열 컬럼 결측 처리
for col in ['fuel', 'seller_type', 'transmission', 'owner', 'brand']:
    car[col] = car[col].astype(str).fillna('Unknown')

# 그래프에서 지나치게 큰 이상치가 전체 형태를 가리지 않도록 기본 분석용 데이터 생성
# 원본 데이터는 유지하되, 시각화는 99백분위 이하 가격을 중심으로 사용한다.
price_upper = car['price_lakh'].quantile(0.99)
km_upper = car['km_driven'].quantile(0.99)
car_viz = car[(car['price_lakh'] <= price_upper) & (car['km_driven'] <= km_upper)].copy()

print('시각화용 중고차 데이터 크기:', car_viz.shape)
display(car_viz.head())

### Bokeh 그래프 1. 연료 종류별 주행거리와 가격 관계

이 그래프의 목적은 연료 종류에 따라 주행거리와 중고차 가격의 관계가 어떻게 달라지는지 확인하는 것이다. 드롭다운에서 연료 종류를 선택하면 해당 연료 차량만 그래프에 표시된다.

X축은 주행거리, Y축은 판매가격이다. 각 점은 차량 한 대를 의미한다. 주행거리가 길수록 가격이 낮아지는 경향이 나타난다면, 주행거리가 중고차 가격에 중요한 영향을 주는 변수라고 볼 수 있다.

In [ ]:
source_all = ColumnDataSource(car_viz)

fuels = sorted(car_viz['fuel'].unique().tolist())
default_fuel = fuels[0]
source_fuel = ColumnDataSource(car_viz[car_viz['fuel'] == default_fuel])

fuel_select = Select(title='연료 종류 선택', value=default_fuel, options=fuels)

p1 = figure(
    width=850,
    height=450,
    title='연료 종류별 주행거리와 중고차 가격 관계',
    x_axis_label='주행거리(km)',
    y_axis_label='판매가격(Lakh)',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p1.scatter(x='km_driven', y='price_lakh', source=source_fuel, size=6, alpha=0.5)

p1.add_tools(HoverTool(tooltips=[
    ('차량명', '@name'),
    ('연료', '@fuel'),
    ('주행거리', '@km_driven{0,0} km'),
    ('가격', '@price_lakh{0.00} Lakh')
]))

callback = CustomJS(args=dict(source_all=source_all, source_fuel=source_fuel, fuel_select=fuel_select), code="""
    const data = source_all.data;
    const selected = fuel_select.value;
    const new_data = {};

    for (const key in data) {
        new_data[key] = [];
    }

    for (let i = 0; i < data['fuel'].length; i++) {
        if (data['fuel'][i] === selected) {
            for (const key in data) {
                new_data[key].push(data[key][i]);
            }
        }
    }

    source_fuel.data = new_data;
    source_fuel.change.emit();
""")

fuel_select.js_on_change('value', callback)
show(column(fuel_select, p1))

### Bokeh 그래프 2. 제조사별 제조연도와 가격 분포

이 그래프의 목적은 제조사별로 제조연도와 가격의 관계를 확인하는 것이다. 드롭다운에서 제조사를 선택하면 해당 제조사의 차량만 표시된다.

X축은 제조연도, Y축은 판매가격이다. 같은 제조사 안에서도 연식이 최신일수록 가격이 높게 유지되는지 확인할 수 있다.

In [ ]:
brand_counts = car_viz['brand'].value_counts()
brands = sorted(brand_counts[brand_counts >= 20].index.tolist())
default_brand = brands[0]

source_brand = ColumnDataSource(car_viz[car_viz['brand'] == default_brand])
brand_select = Select(title='제조사 선택', value=default_brand, options=brands)

p2 = figure(
    width=850,
    height=450,
    title='제조사별 제조연도와 중고차 가격 분포',
    x_axis_label='제조연도',
    y_axis_label='판매가격(Lakh)',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p2.scatter(x='year', y='price_lakh', source=source_brand, size=7, alpha=0.6)

p2.add_tools(HoverTool(tooltips=[
    ('차량명', '@name'),
    ('제조사', '@brand'),
    ('연도', '@year'),
    ('가격', '@price_lakh{0.00} Lakh'),
    ('주행거리', '@km_driven{0,0} km')
]))

callback = CustomJS(args=dict(source_all=source_all, source_brand=source_brand, brand_select=brand_select), code="""
    const data = source_all.data;
    const selected = brand_select.value;
    const new_data = {};

    for (const key in data) {
        new_data[key] = [];
    }

    for (let i = 0; i < data['brand'].length; i++) {
        if (data['brand'][i] === selected) {
            for (const key in data) {
                new_data[key].push(data[key][i]);
            }
        }
    }

    source_brand.data = new_data;
    source_brand.change.emit();
""")

brand_select.js_on_change('value', callback)
show(column(brand_select, p2))

### Bokeh 그래프 3. 제조연도 범위에 따른 연도별 평균 가격

이 그래프의 목적은 제조연도에 따라 평균 중고차 가격이 어떻게 변하는지 확인하는 것이다. 슬라이더를 조절하면 원하는 제조연도 범위만 선택할 수 있다.

X축은 제조연도, Y축은 평균 판매가격이다. 최근 연식일수록 평균 가격이 높게 나타난다면 제조연도가 중고차 가격에 큰 영향을 준다고 해석할 수 있다.

In [ ]:
year_avg = (
    car_viz.groupby('year')['price_lakh']
    .mean()
    .reset_index()
    .sort_values('year')
)

source_year_all = ColumnDataSource(year_avg)
source_year = ColumnDataSource(year_avg)

min_year = int(year_avg['year'].min())
max_year = int(year_avg['year'].max())

year_slider = RangeSlider(
    title='제조연도 범위 선택',
    start=min_year,
    end=max_year,
    value=(min_year, max_year),
    step=1
)

p3 = figure(
    width=850,
    height=450,
    title='제조연도별 평균 중고차 가격',
    x_axis_label='제조연도',
    y_axis_label='평균 판매가격(Lakh)',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p3.line(x='year', y='price_lakh', source=source_year, line_width=3)
p3.scatter(x='year', y='price_lakh', source=source_year, size=7)

p3.add_tools(HoverTool(tooltips=[
    ('제조연도', '@year'),
    ('평균 가격', '@price_lakh{0.00} Lakh')
]))

callback = CustomJS(args=dict(source_all=source_year_all, source_year=source_year, year_slider=year_slider), code="""
    const data = source_all.data;
    const low = year_slider.value[0];
    const high = year_slider.value[1];

    const new_data = {year: [], price_lakh: []};

    for (let i = 0; i < data['year'].length; i++) {
        if (data['year'][i] >= low && data['year'][i] <= high) {
            new_data['year'].push(data['year'][i]);
            new_data['price_lakh'].push(data['price_lakh'][i]);
        }
    }

    source_year.data = new_data;
    source_year.change.emit();
""")

year_slider.js_on_change('value', callback)
show(column(year_slider, p3))

### Bokeh 그래프 4. 변속기 선택에 따른 제조사별 평균 가격 TOP 10

이 그래프의 목적은 자동 변속기와 수동 변속기 차량에서 어떤 제조사의 평균 가격이 높은지 비교하는 것이다. 드롭다운에서 변속기 종류를 선택하면 해당 변속기 차량의 제조사별 평균 가격 TOP 10이 표시된다.

X축은 제조사, Y축은 평균 판매가격이다. 이 그래프를 통해 변속기 조건에 따라 가격대가 높은 제조사가 어떻게 달라지는지 확인할 수 있다.

In [ ]:
brand_trans_summary = (
    car_viz.groupby(['transmission', 'brand'])
    .agg(avg_price_lakh=('price_lakh', 'mean'), count=('price_lakh', 'size'))
    .reset_index()
)
brand_trans_summary = brand_trans_summary[brand_trans_summary['count'] >= 10]
brand_trans_summary = brand_trans_summary.sort_values(['transmission', 'avg_price_lakh'], ascending=[True, False])
brand_trans_summary['rank'] = brand_trans_summary.groupby('transmission').cumcount() + 1
brand_trans_summary = brand_trans_summary[brand_trans_summary['rank'] <= 10]

transmissions = sorted(brand_trans_summary['transmission'].unique().tolist())
default_trans = transmissions[0]
initial_trans = brand_trans_summary[brand_trans_summary['transmission'] == default_trans]

source_trans_all = ColumnDataSource(brand_trans_summary)
source_trans = ColumnDataSource(initial_trans)

trans_select = Select(title='변속기 선택', value=default_trans, options=transmissions)

p4 = figure(
    x_range=initial_trans['brand'].tolist(),
    width=850,
    height=450,
    title='변속기별 제조사 평균 가격 TOP 10',
    x_axis_label='제조사',
    y_axis_label='평균 판매가격(Lakh)',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p4.vbar(x='brand', top='avg_price_lakh', source=source_trans, width=0.6)
p4.xaxis.major_label_orientation = 0.8

p4.add_tools(HoverTool(tooltips=[
    ('변속기', '@transmission'),
    ('제조사', '@brand'),
    ('평균 가격', '@avg_price_lakh{0.00} Lakh'),
    ('차량 수', '@count')
]))

callback = CustomJS(args=dict(source_all=source_trans_all, source_trans=source_trans, trans_select=trans_select, p=p4), code="""
    const data = source_all.data;
    const selected = trans_select.value;
    const new_data = {transmission: [], brand: [], avg_price_lakh: [], count: [], rank: []};

    for (let i = 0; i < data['transmission'].length; i++) {
        if (data['transmission'][i] === selected) {
            new_data['transmission'].push(data['transmission'][i]);
            new_data['brand'].push(data['brand'][i]);
            new_data['avg_price_lakh'].push(data['avg_price_lakh'][i]);
            new_data['count'].push(data['count'][i]);
            new_data['rank'].push(data['rank'][i]);
        }
    }

    source_trans.data = new_data;
    p.x_range.factors = new_data['brand'];
    source_trans.change.emit();
""")

trans_select.js_on_change('value', callback)
show(column(trans_select, p4))

### Bokeh 그래프 5. 소유자 유형별 제조연도와 가격 분포

이 그래프의 목적은 소유자 유형에 따라 차량 가격 분포가 어떻게 달라지는지 확인하는 것이다. 드롭다운에서 소유자 유형을 선택하면 해당 차량만 표시된다.

X축은 제조연도, Y축은 판매가격이다. 일반적으로 첫 번째 소유 차량은 가격이 높게 유지될 가능성이 있고, 여러 번 소유자가 바뀐 차량은 가격이 낮아질 수 있다.

In [ ]:
owners = sorted(car_viz['owner'].unique().tolist())
default_owner = owners[0]
source_owner = ColumnDataSource(car_viz[car_viz['owner'] == default_owner])

owner_select = Select(title='소유자 유형 선택', value=default_owner, options=owners)

p5 = figure(
    width=850,
    height=450,
    title='소유자 유형별 제조연도와 가격 분포',
    x_axis_label='제조연도',
    y_axis_label='판매가격(Lakh)',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p5.scatter(x='year', y='price_lakh', source=source_owner, size=7, alpha=0.55)

p5.add_tools(HoverTool(tooltips=[
    ('차량명', '@name'),
    ('소유자 유형', '@owner'),
    ('제조연도', '@year'),
    ('가격', '@price_lakh{0.00} Lakh'),
    ('주행거리', '@km_driven{0,0} km')
]))

callback = CustomJS(args=dict(source_all=source_all, source_owner=source_owner, owner_select=owner_select), code="""
    const data = source_all.data;
    const selected = owner_select.value;
    const new_data = {};

    for (const key in data) {
        new_data[key] = [];
    }

    for (let i = 0; i < data['owner'].length; i++) {
        if (data['owner'][i] === selected) {
            for (const key in data) {
                new_data[key].push(data[key][i]);
            }
        }
    }

    source_owner.data = new_data;
    source_owner.change.emit();
""")

owner_select.js_on_change('value', callback)
show(column(owner_select, p5))

### Bokeh 그래프 6. 가격 범위 선택에 따른 차량 분포

이 그래프의 목적은 사용자가 원하는 가격대의 차량만 선택하여 주행거리와 제조연도 분포를 확인하는 것이다. 가격 범위 슬라이더를 조절하면 해당 가격 범위에 속하는 차량만 표시된다.

X축은 주행거리, Y축은 제조연도이다. 낮은 가격대 차량이 오래된 연식이나 긴 주행거리에 몰려 있는지 확인할 수 있다.

In [ ]:
min_price = float(car_viz['price_lakh'].min())
max_price = float(car_viz['price_lakh'].max())

source_price = ColumnDataSource(car_viz)

price_slider = RangeSlider(
    title='가격 범위 선택(Lakh)',
    start=round(min_price, 1),
    end=round(max_price, 1),
    value=(round(min_price, 1), round(max_price, 1)),
    step=0.5
)

p6 = figure(
    width=850,
    height=450,
    title='가격 범위에 따른 주행거리와 제조연도 분포',
    x_axis_label='주행거리(km)',
    y_axis_label='제조연도',
    tools='pan,wheel_zoom,box_zoom,reset,save'
)

p6.scatter(x='km_driven', y='year', source=source_price, size=6, alpha=0.45)

p6.add_tools(HoverTool(tooltips=[
    ('차량명', '@name'),
    ('가격', '@price_lakh{0.00} Lakh'),
    ('주행거리', '@km_driven{0,0} km'),
    ('제조연도', '@year')
]))

callback = CustomJS(args=dict(source_all=source_all, source_price=source_price, price_slider=price_slider), code="""
    const data = source_all.data;
    const low = price_slider.value[0];
    const high = price_slider.value[1];
    const new_data = {};

    for (const key in data) {
        new_data[key] = [];
    }

    for (let i = 0; i < data['price_lakh'].length; i++) {
        if (data['price_lakh'][i] >= low && data['price_lakh'][i] <= high) {
            for (const key in data) {
                new_data[key].push(data[key][i]);
            }
        }
    }

    source_price.data = new_data;
    source_price.change.emit();
""")

price_slider.js_on_change('value', callback)
show(column(price_slider, p6))

## 최종 결론

스마트폰 데이터에서는 일일 스마트폰 사용 시간, 수면 시간, 스트레스 수준, 중독 여부를 중심으로 시각화하였다. 그래프를 통해 스마트폰 사용 시간이 길어질수록 중독 수준이나 생활 영향과 관련될 수 있음을 확인할 수 있었다. 특히 중독 수준별 사용 시간 비교, 스트레스 수준별 중독 비율, 수면 시간과 사용 시간의 관계가 데이터의 특징을 잘 보여주었다.

중고차 데이터에서는 차량 가격이 제조연도, 주행거리, 연료 종류, 변속기, 소유자 유형 등에 따라 달라지는 모습을 Bokeh 동적 그래프로 확인하였다. 사용자가 직접 연료 종류, 제조사, 제조연도 범위, 변속기, 소유자 유형, 가격 범위를 선택할 수 있게 하여 조건별 가격 변화를 더 쉽게 비교할 수 있도록 하였다.

이번 과제를 통해 데이터 시각화는 단순히 그래프를 많이 그리는 것이 아니라, 데이터에서 확인하고 싶은 질문을 먼저 정하고 그 질문에 맞는 그래프 형태를 선택하는 과정이라는 점을 확인하였다.